# Notebook 2: Data Preprocessing & Optimization
**Goal:** Clean and optimize data for distributed ML.

**Depends on:** Notebook 1 having written `data/amazon_reviews.parquet`

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when
from pyspark.ml.feature import Imputer
import os


In [2]:
#import os
#os.environ['HADOOP_HOME'] = r'C:\hadoop'
#os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')
#os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 8g pyspark-shell'

import os, subprocess
os.environ["JAVA_HOME"] = subprocess.check_output(
    ["/usr/libexec/java_home", "-v", "17"]
).decode().strip()

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('Amazon_Preprocessing') \
    .config('spark.driver.memory', '8g') \
    .config('spark.driver.maxResultSize', '4g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()
print('Spark ready. UI: http://localhost:4040')


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 13:18:53 WARN Utils: Your hostname, Bavishnas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.11 instead (on interface en0)
26/05/06 13:18:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 13:18:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/06 13:18:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark ready. UI: http://localhost:4040


In [3]:
import os, glob
parquet_path = 'data/amazon_reviews.parquet'
abs_path = 'file:///' + os.path.abspath(parquet_path).replace('\\', '/')
df = spark.read.parquet(abs_path)
print(f'Loaded {df.count():,} rows. Schema:')
df.printSchema()


Loaded 25,437,560 rows. Schema:
root
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- small_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- attachment_type: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



# Step 1: Null counts per column (isnan only applies to numeric types)
from pyspark.sql.functions import isnan, isnull, col
from pyspark.sql.types import DoubleType, FloatType

numeric_types = (DoubleType, FloatType)
numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, numeric_types)]

print('=== Null / NaN Counts per Column ===')
for c in df.columns:
    if c in numeric_cols:
        n = df.filter(isnull(col(c)) | isnan(col(c))).count()
    else:
        n = df.filter(isnull(col(c))).count()
    print(f'  {c}: {n:,} nulls')


In [4]:
# Fill nulls in key fields (matched to actual Amazon Reviews 2023 schema)
df_clean = df.fillna({
    'rating': '3',
    'title': 'Unknown',
    'text': '',
    'helpful_vote': 0,
    'verified_purchase': False
})

# Drop rows with no user_id or parent_asin - essential for ALS
df_clean = df_clean.dropna(subset=['user_id', 'parent_asin', 'rating'])

# Cast rating to float for ALS
from pyspark.sql.functions import col as F_col
df_clean = df_clean.withColumn('rating', F_col('rating').cast('float'))

print(f'Clean row count: {df_clean.count():,}')
df_clean.printSchema()


Clean row count: 25,437,560
root
 |-- rating: float (nullable = true)
 |-- title: string (nullable = false)
 |-- text: string (nullable = false)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- small_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- attachment_type: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = false)
 |-- verified_purchase: boolean (nullable = false)



In [5]:
# Fill nulls in key fields
df_clean = df.fillna({
    'rating': 3.0,
    'title': 'Unknown',
    'text': ''
})
print('Nulls filled. Row count:', df_clean.count())


Nulls filled. Row count: 25437560


## Step 2: Cold Start Filtering (min 5 reviews)

In [6]:
# Count reviews per user and per product
user_counts = df_clean.groupBy('user_id').count().withColumnRenamed('count', 'user_review_count')
product_counts = df_clean.groupBy('parent_asin').count().withColumnRenamed('count', 'product_review_count')

# Join and filter out cold-start users/products
df_filtered = df_clean \
    .join(user_counts, 'user_id') \
    .join(product_counts, 'parent_asin') \
    .filter((col('user_review_count') >= 5) & (col('product_review_count') >= 5))

print(f'Before cold-start filter: {df_clean.count():,}')
print(f'After cold-start filter:  {df_filtered.count():,}')


Before cold-start filter: 25,437,560


After cold-start filter:  6,284,735


## Step 3: Save Optimized Parquet (partitioned by rating)

In [8]:
# Save cleaned/filtered data as Parquet (Spark write — no driver collect; avoids maxResultSize)
out_dir = 'data/amazon_clean.parquet'
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, 'clean_data.parquet')
(
    df_filtered.write.mode('overwrite')
    .option('compression', 'snappy')
    .parquet(out_path)
)
print(f'Wrote Parquet dataset to {out_path} (readable via spark.read.parquet; notebook 3 compatible)')


Wrote Parquet dataset to data/amazon_clean.parquet/clean_data.parquet (readable via spark.read.parquet; notebook 3 compatible)
